# InkMind — TinyGPT vs GPT-2: Comparison & Integration

**Owner:** Member 5 (Comparison & Integration)

This notebook brings the project together and compares our from-scratch
**TinyGPT** (`01_TinyGPT.ipynb`) with pretrained **GPT-2** (`GPT2_Exploration_(1).ipynb`,
generation in `04_GPT2_Generation.ipynb`).

```
                    Input prompt
                         |
             +-----------+-----------+
             v                       v
          TinyGPT                  GPT-2          same generate() loop
       (characters)            (BPE subwords)     same sampling code
             |                       |
          decode                  decode          compare TEXT, not token IDs
             +-----------+-----------+
                         v
                 Comparison engine
          +--------------+---------------+
          v              v               v
        Speed      Output quality   Model statistics
                         |
                         v
                 Comparison results
```

### Methodology (read before the numbers)

- **Same loop, same sampler.** Both models run through `inkmind.generate()`:
  the same temperature / top-k / top-p code, the same seeds, and the same
  device. Each step recomputes the forward pass (no KV cache) for both models.
- **Different tokenizers.** TinyGPT predicts characters (vocab 288), GPT-2
  predicts BPE subwords (vocab 50,257). Token IDs and token counts are **not**
  comparable, so outputs are compared as **decoded text**.
- **Equal text budget.** Each model generates until its output reaches
  **200 characters** (or GPT-2 emits `<|endoftext|>`).
- **Timing definitions.** `generation_time_s` = model inference + sampling
  only. Tokenization and decoding are timed separately. `chars_per_s` is
  comparable across models; `tokens_per_s` is not.
- **Different training.** TinyGPT: ~288K parameters trained for 1,000 steps on
  poetry. GPT-2: ~124M parameters pretrained on web text, **not** fine-tuned on
  poetry. This is an educational experiment on a small set of prompts, not a
  benchmark — it does not show that one model is "better" in general.

## 1. Setup

In [ ]:
# --- Setup: works in Google Colab and locally (VS Code / Jupyter) ---------
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nalinegwizaigihozo/Ink-mind_group2.git"

if IN_COLAB:
    !pip install -q transformers
    REPO_DIR = "/content/Ink-mind_group2"
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    # If cloning fails (e.g. private repo), upload the `inkmind/` folder and
    # `data/PoetryFoundationData.csv` into /content/Ink-mind_group2 manually.
else:
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Running in Colab:", IN_COLAB)
print("Repository dir  :", REPO_DIR)

In [ ]:
import textwrap

import torch
import pandas as pd
import matplotlib.pyplot as plt

from inkmind import (CHECKPOINT_NAME, GPT2Adapter, TinyGPTAdapter, find_checkpoint, generate,
                     load_gpt2, load_poetry_corpus, load_poetry_poems, load_tinygpt,
                     save_tinygpt, train_tinygpt)
from inkmind.comparison import (DEFAULT_CONFIGS, build_reference_vocabulary, cross_sample_diversity,
                                environment_info, measure, model_statistics, qualitative_rating_template,
                                quality_summary, run_comparison, speed_summary)
from inkmind.plotting import plot_metric_vs_param

pd.set_option("display.max_colwidth", 120)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = environment_info()
ENV

## 2. Load TinyGPT

The checkpoint saved at the end of `01_TinyGPT.ipynb` is
`inkmind_checkpoint_1000_steps.pth`. It is not stored in GitHub (`*.pth` is
in `.gitignore`), so in Colab either:

- set `UPLOAD_CHECKPOINT = True` and upload the file, **or**
- put it in `/content` or `/content/drive/MyDrive`, **or**
- leave `RETRAIN_IF_MISSING = True`: TinyGPT is retrained with the **same
  recipe** as `01_TinyGPT.ipynb` (same cleaning, vocab, 90/10 split, AdamW 3e-4,
  batch 32, block 256, 1,000 steps). The original notebook did not fix a seed,
  so a retrained model is equivalent but not identical.

Either way, the model is then **loaded from the checkpoint file** so that the
load time is measured the same way as for GPT-2.

In [ ]:
UPLOAD_CHECKPOINT = False
RETRAIN_IF_MISSING = True

DATA_PATH = os.path.join(REPO_DIR, "data", "PoetryFoundationData.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = "PoetryFoundationData.csv"   # uploaded manually, as in 01_TinyGPT.ipynb

tiny_checkpoint = find_checkpoint()

if tiny_checkpoint is None and IN_COLAB and UPLOAD_CHECKPOINT:
    from google.colab import files
    files.upload()
    tiny_checkpoint = find_checkpoint()

retrain_info = None
if tiny_checkpoint is None:
    if not RETRAIN_IF_MISSING:
        raise FileNotFoundError(f"{CHECKPOINT_NAME} not found; upload it or set RETRAIN_IF_MISSING = True")
    print("No checkpoint found -> retraining TinyGPT with the 01_TinyGPT.ipynb recipe\n")
    corpus = load_poetry_corpus(DATA_PATH)
    m, tok, opt, retrain_info = train_tinygpt(corpus, steps=1000, device=device)
    tiny_checkpoint = CHECKPOINT_NAME
    save_tinygpt(tiny_checkpoint, m, tok, opt, training_steps=1000)
    del m, tok, opt, corpus

(tiny_model, tiny_tokenizer, tiny_info), tiny_load_stats = measure(load_tinygpt, tiny_checkpoint, device=device)
if retrain_info:
    tiny_info["source"] = retrain_info["source"]
    print(f"Retraining took {retrain_info['train_time_s']:.1f}s, validation loss {retrain_info['val_loss']:.4f}")

tiny = TinyGPTAdapter(tiny_model, tiny_tokenizer)
print("TinyGPT source:", tiny_info["source"])
print("Load stats    :", {k: round(v, 3) for k, v in tiny_load_stats.items()})

## 3. Load GPT-2

The first call downloads and caches the weights (not timed). The second call
is the timed, warm load, which is what a user of a deployed app would see.

In [ ]:
_warm = load_gpt2("gpt2", device=torch.device("cpu"))   # download / cache only
del _warm

(gpt2_model, gpt2_tokenizer, gpt2_info), gpt2_load_stats = measure(load_gpt2, "gpt2", device=device)
gpt2 = GPT2Adapter(gpt2_model, gpt2_tokenizer)

print("GPT-2 source:", gpt2_info["source"])
print("Load stats  :", {k: round(v, 3) for k, v in gpt2_load_stats.items()})

## 4. Model comparison

All values below are read from the loaded model objects (not typed in by
hand). Parameter counts are unique parameters, so GPT-2's tied
`lm_head`/`wte` weight is counted once (matching Task 3: 124,439,808).

Note on TinyGPT's block: its single block applies attention **without** a
residual connection, then `context + FFN(LayerNorm(context))`. GPT-2 uses
Pre-LN with residual connections around both attention and the MLP.

In [ ]:
model_table = model_statistics(tiny_model, tiny_tokenizer, tiny_info, gpt2_model, gpt2_tokenizer, gpt2_info)
tiny_params = int(model_table.loc["Parameters", "TinyGPT"])
gpt2_params = int(model_table.loc["Parameters", "GPT-2"])
print(f"GPT-2 has {gpt2_params / tiny_params:,.0f}x more parameters than TinyGPT")
model_table

### Same text, different tokens

The same prompt becomes very different token sequences — this is why token IDs
and token counts cannot be compared directly.

In [ ]:
PROMPTS = [
    "Artificial intelligence is",
    "The future of technology is",
    "Machine learning can",
    "Once upon a time",
    "Love is a beautiful thing",          # prompt used in 01_TinyGPT.ipynb
    "The moon shines over the hills",     # prompt used in Tasks 1-3
]

rows = []
for p in PROMPTS:
    t_ids, g_ids = tiny.encode(p), gpt2.encode(p)
    rows.append({"prompt": p, "characters": len(p),
                 "TinyGPT tokens": len(t_ids), "GPT-2 tokens": len(g_ids),
                 "TinyGPT IDs (first 8)": t_ids[:8], "GPT-2 IDs": g_ids})
pd.DataFrame(rows).set_index("prompt")

## 5. Generation comparison

Every prompt × every decoding setting × 3 seeds (greedy once, since it is
deterministic), with a budget of 200 generated characters.

Settings (`inkmind.comparison.DEFAULT_CONFIGS`): greedy, sampling T=1.0,
sampling T=0.7, top-k=40 (T=0.8), top-p=0.9 (T=0.8).

In [ ]:
SEEDS = (0, 1, 2)
MAX_NEW_CHARS = 200

# Reference vocabulary for the known-word proxy (words that occur in the poetry corpus)
reference_vocab = build_reference_vocabulary(load_poetry_poems(DATA_PATH))
print("Reference vocabulary:", f"{len(reference_vocab):,}", "distinct words\n")

# Warm-up (first CUDA / kernel calls are slower and should not be timed)
for adapter in (tiny, gpt2):
    generate(adapter, "warm up", max_new_tokens=5)

results = run_comparison([tiny, gpt2], PROMPTS, DEFAULT_CONFIGS, seeds=SEEDS,
                         max_new_chars=MAX_NEW_CHARS, reference_vocab=reference_vocab, verbose=False)
print(f"{len(results)} generations on {device.type}")

### 5.1 Side-by-side outputs (top-p = 0.9, T = 0.8, seed 0)

With the same seed, a model draws the same random numbers for every prompt,
so a weak model's outputs for different prompts can look alike. Compare the
other seeds in `results` before drawing conclusions from one sample.

In [ ]:
def side_by_side(df, config="top-p=0.9 T=0.8", seed=0, width=100):
    sel = df[(df["config"] == config) & (df["seed"] == seed)]
    for prompt in PROMPTS:
        print("=" * width)
        print(f"PROMPT: {prompt!r}   [{config}, seed {seed}]")
        for model_name in ["TinyGPT", "GPT-2"]:
            row = sel[(sel["model"] == model_name) & (sel["prompt"] == prompt)].iloc[0]
            text = textwrap.fill(row["completion"].replace("\n", " ⏎ "), width - 4,
                                 initial_indent="    ", subsequent_indent="    ")
            print(f"\n  {model_name}  ({row['generated_tokens']} tokens, {row['generated_chars']} chars, "
                  f"{row['generation_time_s']:.2f}s, stop: {row['stop_reason']})")
            print(text)
        print()

side_by_side(results)

### 5.2 Greedy outputs (deterministic)

In [ ]:
side_by_side(results, config="greedy")

## 6. Speed comparison

Means over all generations above. `generation_time_s` covers model inference +
sampling; tokenization and decoding are listed separately.

In [ ]:
speed = speed_summary(results)
speed

In [ ]:
# Controlled latency benchmark: same prompt, greedy, 200 characters, 3 repeats per model
bench_rows = []
for adapter in (tiny, gpt2):
    for rep in range(3):
        r, stats = measure(generate, adapter, "Once upon a time", strategy="greedy",
                           max_new_tokens=MAX_NEW_CHARS, max_new_chars=MAX_NEW_CHARS)
        bench_rows.append({"model": adapter.name, "repeat": rep, "generated_tokens": r.generated_tokens,
                           "generated_chars": r.generated_chars, "latency_s": stats["wall_time_s"],
                           "generation_time_s": r.generation_time_s,
                           "ms_per_token": 1000 * r.generation_time_s / r.generated_tokens,
                           "chars_per_s": r.chars_per_s,
                           "cpu_utilisation": stats.get("cpu_utilisation", float("nan")),
                           "gpu_peak_memory_mb": stats.get("gpu_peak_memory_mb", float("nan"))})
bench = pd.DataFrame(bench_rows)
bench_summary = bench.groupby("model").mean(numeric_only=True).drop(columns="repeat").round(4)
bench_summary

**How to read this:** a TinyGPT step is far cheaper than a GPT-2 step (≈430x
fewer parameters), but TinyGPT needs several times more steps for the same amount of
text because each step yields one character (compare `generated_tokens`
for the same 200-character budget). `chars_per_s` combines both
effects and is the fair speed figure.

## 7. Resource comparison

In [ ]:
def fmt(v, digits=3):
    return "unavailable" if v is None or v != v else round(v, digits)

has_gpu = device.type == "cuda"
resources = pd.DataFrame([
    ("Device", device.type, device.type, "same device for both"),
    ("Parameter memory (MB, fp32)", model_table.loc["Parameter memory (MB, fp32)", "TinyGPT"],
     model_table.loc["Parameter memory (MB, fp32)", "GPT-2"], "measured from parameters"),
    ("Load time, warm (s)", fmt(tiny_load_stats["wall_time_s"]), fmt(gpt2_load_stats["wall_time_s"]),
     "measured, single run"),
    ("Process RSS increase while loading (MB)", fmt(tiny_load_stats.get("rss_delta_mb"), 1),
     fmt(gpt2_load_stats.get("rss_delta_mb"), 1), "indicative only (noisy)"),
    ("GPU peak memory while loading (MB)",
     fmt(tiny_load_stats.get("gpu_peak_memory_mb"), 1) if has_gpu else "unavailable (no GPU)",
     fmt(gpt2_load_stats.get("gpu_peak_memory_mb"), 1) if has_gpu else "unavailable (no GPU)", "measured"),
    ("Latency for 200 chars, greedy (s)", fmt(bench_summary.loc["TinyGPT", "latency_s"]),
     fmt(bench_summary.loc["GPT-2", "latency_s"]), "measured, mean of 3"),
    ("CPU utilisation during generation", fmt(bench_summary.loc["TinyGPT", "cpu_utilisation"], 2),
     fmt(bench_summary.loc["GPT-2", "cpu_utilisation"], 2), "process CPU time / wall time"),
    ("GPU peak memory during generation (MB)",
     fmt(bench_summary.loc["TinyGPT", "gpu_peak_memory_mb"], 1) if has_gpu else "unavailable (no GPU)",
     fmt(bench_summary.loc["GPT-2", "gpu_peak_memory_mb"], 1) if has_gpu else "unavailable (no GPU)",
     "measured"),
    ("GPU utilisation (%)", "unavailable", "unavailable", "not measured reliably from inside the notebook"),
], columns=["Metric", "TinyGPT", "GPT-2", "How measured"]).set_index("Metric")
resources

## 8. Output quality — automatic indicators

All indicators are computed on **decoded words**, so they work the same for
both tokenizers:

| Indicator | Meaning | Caveat |
|---|---|---|
| `known_word_rate` | share of generated words that appear in the poetry corpus | a proxy for "produces real words"; biased **towards** TinyGPT (its training data) |
| `distinct_1`, `distinct_2` | unique words / word-pairs ÷ total in one output | variety, not quality; gibberish can score high |
| `repeated_3gram_rate` | share of 3-word sequences already used earlier in the output | detects looping |

None of these measures coherence, relevance or grammar — see section 10.

**Check `n_words` first.** A ratio computed on a handful of words is not
meaningful. For example, an output that is mostly spaces and newlines with a
few words can score a perfect known-word rate.

In [ ]:
quality = quality_summary(results)
quality

### 8.1 Temperature behaviour of both models

In [ ]:
temps = [0.5, 0.8, 1.0, 1.3]
temp_results = run_comparison(
    [tiny, gpt2], PROMPTS, {f"T={t}": dict(strategy="sample", temperature=t) for t in temps},
    seeds=SEEDS, max_new_chars=MAX_NEW_CHARS, reference_vocab=reference_vocab, verbose=False)

fig = plot_metric_vs_param(
    temp_results, "temperature",
    ["known_word_rate", "distinct_2", "repeated_3gram_rate"],
    ["Known-word rate", "Distinct-2", "Repeated 3-gram rate"],
    suptitle=f"Effect of temperature (pure sampling; mean over {len(PROMPTS)} prompts x {len(SEEDS)} seeds)",
)
plt.show()
temp_results.groupby(["model", "temperature"])[["known_word_rate", "distinct_2", "repeated_3gram_rate"]].mean().round(3)

### 8.2 Diversity across seeds

For the same prompt and setting, how different are the outputs from different
seeds? (`mean_pairwise_jaccard_distance`: 0 = identical word-pairs, 1 = no
shared word-pairs.)

In [ ]:
diversity = cross_sample_diversity(results[results["config"] != "greedy"])
diversity_summary = diversity.groupby(["model", "config"])[
    ["unique_outputs", "pooled_distinct_2", "mean_pairwise_jaccard_distance"]].mean().round(3)
diversity_summary

## 9. Comparison results

In [ ]:
def mean_metric(df, model, col):
    return round(df[df["model"] == model][col].mean(), 3)

sampled = results[results["config"] != "greedy"]
div_by_model = diversity.groupby("model")["mean_pairwise_jaccard_distance"].mean()

final = pd.DataFrame([
    ("Parameters", f"{tiny_params:,}", f"{gpt2_params:,}", "measured"),
    ("Vocabulary / tokenizer", f"{tiny_tokenizer.vocab_size} characters", f"{len(gpt2_tokenizer):,} BPE tokens", "measured"),
    ("Context length (tokens)", tiny_model.block_size, gpt2_model.config.n_positions, "measured"),
    ("Layers / heads / embedding", f"{tiny_model.n_layer} / {tiny_model.n_head} / {tiny_model.n_embd}",
     f"{gpt2_model.config.n_layer} / {gpt2_model.config.n_head} / {gpt2_model.config.n_embd}", "measured"),
    ("Characters per second", speed.loc["TinyGPT", "chars_per_s"].round(1), speed.loc["GPT-2", "chars_per_s"].round(1),
     f"measured on {device.type}; comparable"),
    ("Tokens per second", speed.loc["TinyGPT", "tokens_per_s"].round(1), speed.loc["GPT-2", "tokens_per_s"].round(1),
     "measured; NOT comparable (different token units)"),
    ("Latency for 200 chars, greedy (s)", fmt(bench_summary.loc["TinyGPT", "latency_s"]),
     fmt(bench_summary.loc["GPT-2", "latency_s"]), "measured, mean of 3"),
    ("Known-word rate (sampled outputs)", mean_metric(sampled, "TinyGPT", "known_word_rate"),
     mean_metric(sampled, "GPT-2", "known_word_rate"), "automatic proxy"),
    ("Distinct-2 (sampled outputs)", mean_metric(sampled, "TinyGPT", "distinct_2"),
     mean_metric(sampled, "GPT-2", "distinct_2"), "automatic proxy"),
    ("Words per 200-char output (greedy)", mean_metric(results[results["config"] == "greedy"], "TinyGPT", "n_words"),
     mean_metric(results[results["config"] == "greedy"], "GPT-2", "n_words"), "measured; read the ratios below with this"),
    ("Repeated 3-gram rate (greedy)", mean_metric(results[results["config"] == "greedy"], "TinyGPT", "repeated_3gram_rate"),
     mean_metric(results[results["config"] == "greedy"], "GPT-2", "repeated_3gram_rate"), "automatic proxy"),
    ("Cross-seed diversity (Jaccard distance)", round(div_by_model["TinyGPT"], 3), round(div_by_model["GPT-2"], 3),
     "automatic proxy"),
    ("Coherence, relevance, grammar, consistency", "see section 10", "see section 10", "qualitative (human rating)"),
], columns=["Criterion", "TinyGPT", "GPT-2", "Type"]).set_index("Criterion")
final

## 10. Qualitative comparison (human evaluation)

Automatic indicators cannot judge whether text *makes sense*. Read the
side-by-side outputs in section 5 and rate each output from 1 (poor) to 5 (good):

| Criterion | Question to ask |
|---|---|
| **Coherence** | Do the sentences/lines connect logically? |
| **Relevance** | Does the continuation follow the prompt's topic? |
| **Grammar** | Are the words real and the sentences well formed? |
| **Context consistency** | Does the text stay consistent with itself (names, tense, topic)? |
| **Diversity** | Do different seeds give genuinely different continuations? |

The cell below exports a blank rating sheet (one row per output) to fill in as
a team. Averaging ratings from several team members is more reliable than one
person's judgement.

**Evidence already in the project:** the TinyGPT sample in
`01_TinyGPT.ipynb` (section 29, after 1,000 steps, validation loss ≈ 2.49)
continues *"Love is a beautiful thing"* with poem-like line breaks and
punctuation but mostly non-words. Compare that with what you see above for
both models before writing conclusions.

In [ ]:
rating_sheet = qualitative_rating_template(results[results["config"].isin(["greedy", "top-p=0.9 T=0.8"])])
rating_path = os.path.join(RESULTS_DIR, "task5_human_rating_sheet.csv")
rating_sheet.to_csv(rating_path, index=False)
print("Rating sheet with", len(rating_sheet), "outputs saved to", rating_path)
rating_sheet.head()

## 11. Save results

In [ ]:
DOWNLOAD_RESULTS = False   # set True in Colab to download the CSV files

paths = {
    "task5_generation_results.csv": results.assign(run_device=device.type),
    "task5_temperature_results.csv": temp_results.assign(run_device=device.type),
    "task5_model_comparison.csv": model_table,
    "task5_resources.csv": resources,
    "task5_final_comparison.csv": final,
}
for name, df in paths.items():
    path = os.path.join(RESULTS_DIR, name)
    df.to_csv(path, index=not name.endswith("results.csv"))
    print("saved", path)

if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    for name in paths:
        files.download(os.path.join(RESULTS_DIR, name))

## 12. Conclusions and limitations

Write the final conclusions from **your** run's tables above. Points the
comparison is designed to show:

- **Scale:** GPT-2 has roughly 430x more parameters, 12x more layers and
  heads, a 4x longer context and a subword vocabulary. It was pretrained on far
  more (and more general) text than TinyGPT's 1,000 steps on poetry.
- **Tokenization changes the cost of text:** TinyGPT needs one step per
  character, GPT-2 one step per subword, so per-token speed is misleading and
  characters per second is the fair measure.
- **Decoding settings affect both models in the same direction** (temperature
  up → more variety, less repetition), but the size of the effect depends on
  how well trained the model is.
- **Domain:** TinyGPT only saw poetry; GPT-2 here is not fine-tuned on poetry,
  so neither model is a perfect fit for the InkMind goal.

**Limitations:** six prompts and three seeds; automatic metrics are proxies;
timings depend on the hardware (Colab GPU vs CPU) and on the uncached loop;
the TinyGPT checkpoint may be retrained rather than the original. The
comparison is an educational experiment, not a benchmark.